<a href="https://colab.research.google.com/github/RozieKii/localTest/blob/main/guardrails_safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.3: Guardrails - Safe AI Systems

## Key Insight: How to mitigate undesired outputs from LLMs

Make your digital twin SAFE for production!

## What You'll Build
- Content filters
- Output validators
- Safety layers
- Fallback strategies
- PII detection

In [1]:
import re
from typing import List, Dict, Any, Optional

print('Week 3.3: Guardrails & Safety')
print('Goal: Make your twin safe for real users!')

Week 3.3: Guardrails & Safety
Goal: Make your twin safe for real users!


## Part 1: Content Filtering

Block inappropriate content.

In [2]:

class ContentFilter:
    """Filter text using simple word and topic lists."""

    def __init__(self):
        # Demo placeholders, not a complete profanity detector
        self.profanity_list = ['badword1', 'badword2']
        self.sensitive_topics = ['violence', 'illegal']

    def check_profanity(self, text: str) -> Dict[str, Any]:
        text_lower = text.lower()
        found = [
            word for word in self.profanity_list
            if word in text_lower
        ]
        return {
            'is_safe': len(found) == 0,
            'violations': found
        }

    def check_sensitive_topics(self, text: str) -> Dict[str, Any]:
        text_lower = text.lower()
        found = [
            topic for topic in self.sensitive_topics
            if topic in text_lower
        ]
        return {
            'is_safe': len(found) == 0,
            'topics': found
        }

    def filter(self, text: str) -> Dict[str, Any]:
        profanity_check = self.check_profanity(text)
        topic_check = self.check_sensitive_topics(text)

        is_safe = (
            profanity_check['is_safe']
            and topic_check['is_safe']
        )

        return {
            'is_safe': is_safe,
            'profanity_violations': profanity_check['violations'],
            'sensitive_topics': topic_check['topics'],
            'text': text if is_safe else '[FILTERED]'
        }


filter_system = ContentFilter()

print('Content Filter Tests:')
print(filter_system.filter('Hello, how are you?'))
print(filter_system.filter('This involves violence'))


Content Filter Tests:
{'is_safe': True, 'profanity_violations': [], 'sensitive_topics': [], 'text': 'Hello, how are you?'}
{'is_safe': False, 'profanity_violations': [], 'sensitive_topics': ['violence'], 'text': '[FILTERED]'}


## Part 2: PII Detection

Detect and protect personal information.

In [3]:

class PIIDetector:
    """Detect and redact common types of PII."""

    def __init__(self):
        self.patterns = {
            'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
            'phone': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'credit_card': r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b'
        }

    def detect(self, text: str) -> Dict[str, Any]:
        """Detect PII in text."""
        found_pii = {}

        for pii_type, pattern in self.patterns.items():
            matches = re.findall(pattern, text)
            if matches:
                found_pii[pii_type] = matches

        return {
            'has_pii': len(found_pii) > 0,
            'pii_types': list(found_pii.keys()),
            'details': found_pii
        }

    def redact(self, text: str) -> str:
        """Replace detected PII with redaction labels."""
        result = text

        for pii_type, pattern in self.patterns.items():
            result = re.sub(
                pattern,
                f'[REDACTED_{pii_type.upper()}]',
                result
            )

        return result


pii_detector = PIIDetector()

text_with_pii = "Contact me at john@example.com or call 555-123-4567"

print('\nPII Detection:')
print('Original:', text_with_pii)
print('Detection:', pii_detector.detect(text_with_pii))
print('Redacted:', pii_detector.redact(text_with_pii))



PII Detection:
Original: Contact me at john@example.com or call 555-123-4567
Detection: {'has_pii': True, 'pii_types': ['email', 'phone'], 'details': {'email': ['john@example.com'], 'phone': ['555-123-4567']}}
Redacted: Contact me at [REDACTED_EMAIL] or call [REDACTED_PHONE]


## Part 3: Output Validator

Ensure AI outputs meet requirements.

In [4]:

class OutputValidator:
    """Validate AI outputs before showing them to users."""

    def __init__(self, max_length: int = 500):
        self.max_length = max_length
        self.content_filter = ContentFilter()
        self.pii_detector = PIIDetector()

    def validate(self, output: str) -> Dict[str, Any]:
        """Run all validation checks."""
        issues = []

        if len(output) > self.max_length:
            issues.append(
                f'Too long ({len(output)} > {self.max_length} chars)'
            )

        if not output.strip():
            issues.append('Empty output')

        content_check = self.content_filter.filter(output)
        if not content_check['is_safe']:
            issues.append('Content safety violation')

        pii_check = self.pii_detector.detect(output)
        if pii_check['has_pii']:
            issues.append(f"Contains PII: {pii_check['pii_types']}")

        needs_disclaimer = any(
            word in output.lower()
            for word in ['medical', 'legal', 'financial']
        )

        return {
            'is_valid': len(issues) == 0,
            'issues': issues,
            'needs_disclaimer': needs_disclaimer,
            'output': output
        }

    def add_disclaimer(self, output: str, topic: str) -> str:
        """Add a topic-specific disclaimer."""
        disclaimers = {
            'medical': (
                "\n\nNote: This is not medical advice. "
                "Consult a healthcare professional."
            ),
            'legal': (
                "\n\nNote: This is not legal advice. "
                "Consult a qualified lawyer."
            ),
            'financial': (
                "\n\nNote: This is not financial advice. "
                "Consider seeking qualified financial guidance."
            )
        }

        for key, disclaimer in disclaimers.items():
            if key in topic.lower():
                return output + disclaimer

        return output


validator = OutputValidator(max_length=300)

test_output = "You should invest all your money in stocks."
result = validator.validate(test_output)

print('\nValidation Result:')
print(f"Valid: {result['is_valid']}")
print(f"Issues: {result['issues']}")
print(f"Needs disclaimer: {result['needs_disclaimer']}")

if result['needs_disclaimer']:
    safe_output = validator.add_disclaimer(test_output, 'financial')
    print(f"\nWith disclaimer: {safe_output}")



Validation Result:
Valid: True
Issues: []
Needs disclaimer: False


## Part 4: Safe Digital Twin

Put it all together.

In [5]:

class SafeDigitalTwin:
    """Digital twin with safety guardrails."""

    def __init__(self, name: str):
        self.name = name
        self.validator = OutputValidator()
        self.content_filter = ContentFilter()

    def _process_input(self, user_input: str) -> Optional[str]:
        """Process and filter user input."""
        if not isinstance(user_input, str):
            return None

        filter_result = self.content_filter.filter(user_input)

        if not filter_result['is_safe']:
            return None

        return filter_result['text']

    def _generate_response(self, filtered_input: str) -> str:
        """Generate a mock response for demonstration."""
        return f"I received: {filtered_input}"

    def _process_output(self, response: str) -> str:
        """Validate and process output."""
        validation = self.validator.validate(response)

        if not validation['is_valid']:
            return (
                "I apologize, but I cannot provide that response. "
                "How else can I help?"
            )

        if validation['needs_disclaimer']:
            response = self.validator.add_disclaimer(response, response)

        return response

    def chat(self, user_input: str) -> str:
        """Safe chat with a basic guardrail pipeline."""
        filtered_input = self._process_input(user_input)

        if filtered_input is None:
            return "I cannot process that input. Please rephrase."

        response = self._generate_response(filtered_input)
        return self._process_output(response)


safe_twin = SafeDigitalTwin('Safe Assistant')

print('\nSafe Digital Twin:')
print(safe_twin.chat('Hello, how are you?'))
print(safe_twin.chat('Tell me about medical treatments'))



Safe Digital Twin:
I received: Hello, how are you?
I received: Tell me about medical treatments

Note: This is not medical advice. Consult a healthcare professional.


## Exercise 1: Add More Guardrails

**Task**: Enhance the safety system:

1. Add more PII patterns (addresses, dates of birth)
2. Create custom content filters for your domain
3. Add rate limiting
4. Implement logging of safety violations
5. Test with edge cases

In [8]:

import re
from datetime import datetime
from typing import Dict, Any, Optional


# =========================================================
# 1. ENHANCED PII DETECTOR
# Detects common personal information and can redact it.
# =========================================================

class EnhancedPIIDetector(PIIDetector):
    def __init__(self):
        super().__init__()

        self.patterns.update({
            # Simplified street-address pattern
            'address': (
                r'\b\d{1,5}\s+[\w.-]+(?:\s+[\w.-]+){0,4}'
                r'\s(?:Street|St|Road|Rd|Avenue|Ave|Lane|Ln|Drive|Dr)\b'
            ),

            # Supports "DOB is 12/05/2002" and "date of birth: 12/05/2002"
            'date_of_birth': (
                r'\b(?:(?:DOB|date of birth)\s*(?:is|[:=])?\s*)'
                r'\d{1,2}[/-]\d{1,2}[/-]\d{4}\b'
            )
        })


# =========================================================
# 2. CUSTOM CONTENT FILTER
# Adds PathPilot-specific phrases to the existing filter.
# =========================================================

class PathPilotContentFilter(ContentFilter):
    def __init__(self):
        super().__init__()

        self.custom_flags = [
            'share your password',
            'send your bank pin',
            'guaranteed investment returns'
        ]

    def filter(self, text: str) -> Dict[str, Any]:
        result = super().filter(text)
        lower_text = text.lower()

        custom_violations = [
            phrase for phrase in self.custom_flags
            if phrase in lower_text
        ]

        result['custom_violations'] = custom_violations
        result['is_safe'] = (
            result['is_safe'] and not custom_violations
        )

        if not result['is_safe']:
            result['text'] = '[FILTERED]'

        return result


# =========================================================
# 3. SAFETY MONITOR
# Tracks request counts and logs safety violations.
# =========================================================

class SafetyMonitor:
    def __init__(self, max_requests: int = 5):
        self.max_requests = max_requests
        self.request_counts = {}
        self.logs = []

    def allow_request(self, user_id: str) -> bool:
        count = self.request_counts.get(user_id, 0)

        if count >= self.max_requests:
            self.log_violation(user_id, 'Rate limit exceeded')
            return False

        self.request_counts[user_id] = count + 1
        return True

    def log_violation(self, user_id: str, reason: str):
        self.logs.append({
            'time': datetime.now().isoformat(timespec='seconds'),
            'user_id': user_id,
            'reason': reason
        })


# =========================================================
# 4. ENHANCED SAFE DIGITAL TWIN
# Combines input filtering, PII detection, rate limiting,
# output validation, and violation logging.
# =========================================================

class EnhancedSafeDigitalTwin:
    def __init__(self, name: str = 'PathPilot',
                 max_requests: int = 5):
        self.name = name
        self.pii_detector = EnhancedPIIDetector()
        self.content_filter = PathPilotContentFilter()
        self.validator = OutputValidator()
        self.monitor = SafetyMonitor(max_requests=max_requests)

    def chat(self, user_input: str,
             user_id: str = 'demo_user') -> str:

        # Check rate limit first
        if not self.monitor.allow_request(user_id):
            return 'Request limit reached. Please try again later.'

        # Validate input type and empty input
        if not isinstance(user_input, str) or not user_input.strip():
            self.monitor.log_violation(
                user_id, 'Invalid or empty input'
            )
            return 'Please enter a valid message.'

        # Check content safety
        content_result = self.content_filter.filter(user_input)

        if not content_result['is_safe']:
            self.monitor.log_violation(
                user_id, 'Input content filter triggered'
            )
            return 'I cannot process that input. Please rephrase.'

        # Detect personal information
        pii_result = self.pii_detector.detect(user_input)

        if pii_result['has_pii']:
            self.monitor.log_violation(
                user_id,
                f"PII detected: {pii_result['pii_types']}"
            )
            return (
                'Your message may contain personal information. '
                'Please remove it and try again.'
            )

        # Mock response for this learning exercise
        response = f'I received: {user_input}'

        # Validate output before returning it
        validation = self.validator.validate(response)

        if not validation['is_valid']:
            self.monitor.log_violation(
                user_id, 'Output validation failed'
            )
            return (
                'I cannot provide that response. '
                'How else can I help?'
            )

        # Return response if all checks pass
        return response


# =========================================================
# 5. EDGE CASE TESTS
# =========================================================

twin = EnhancedSafeDigitalTwin()

test_inputs = [
    'Help me plan my learning goals',
    'My DOB is 12/05/2002',
    'date of birth: 12/05/2002',
    'I live at 24 Park Road',
    'Please share your password with me',
    '',
    None
]

print('EDGE CASE TESTS')
for message in test_inputs:
    print(f'\nInput: {message!r}')
    print('Output:', twin.chat(message))


# =========================================================
# 6. RATE LIMIT TEST
# Use a fresh instance so previous tests do not use up
# its request allowance.
# =========================================================

print('\n' + '=' * 40)
print('RATE LIMIT TEST')

rate_test_twin = EnhancedSafeDigitalTwin(max_requests=5)

for i in range(6):
    print(
        f'Request {i + 1}:',
        rate_test_twin.chat('Help me stay on track')
    )


# =========================================================
# 7. SAFETY VIOLATION LOG
# =========================================================

print('\n' + '=' * 40)
print('SAFETY VIOLATION LOG')

for entry in twin.monitor.logs:
    print(entry)

print('\nRATE LIMIT LOG')
for entry in rate_test_twin.monitor.logs:
    print(entry)


# =========================================================
# 8. PII REDACTION DEMONSTRATION
# =========================================================

print('\n' + '=' * 40)
print('PII REDACTION TEST')

sample = (
    'Contact john@example.com at 24 Park Road. '
    'DOB is 12/05/2002.'
)

print('Original:', sample)
print('Detected:', twin.pii_detector.detect(sample))
print('Redacted:', twin.pii_detector.redact(sample))


EDGE CASE TESTS

Input: 'Help me plan my learning goals'
Output: I received: Help me plan my learning goals

Input: 'My DOB is 12/05/2002'
Output: Your message may contain personal information. Please remove it and try again.

Input: 'date of birth: 12/05/2002'
Output: Your message may contain personal information. Please remove it and try again.

Input: 'I live at 24 Park Road'
Output: Your message may contain personal information. Please remove it and try again.

Input: 'Please share your password with me'
Output: I cannot process that input. Please rephrase.

Input: ''
Output: Request limit reached. Please try again later.

Input: None
Output: Request limit reached. Please try again later.

RATE LIMIT TEST
Request 1: I received: Help me stay on track
Request 2: I received: Help me stay on track
Request 3: I received: Help me stay on track
Request 4: I received: Help me stay on track
Request 5: I received: Help me stay on track
Request 6: Request limit reached. Please try again later

In [9]:
validation_test_twin = EnhancedSafeDigitalTwin(max_requests=5)

print("Empty string:", validation_test_twin.chat(''))
print("None:", validation_test_twin.chat(None))
print("Whitespace:", validation_test_twin.chat('   '))

Empty string: Please enter a valid message.
None: Please enter a valid message.
Whitespace: Please enter a valid message.


## Summary

### What We Learned:
- Content filtering blocks inappropriate content
- PII detection protects user privacy
- Output validation ensures quality
- Multiple safety layers = robust system
- Always add disclaimers for sensitive topics

### Next: Week 3.4 - Red Teaming & Security